In [1]:
import re
import nltk
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
THRESHOLD = 0.75 
NGRAM_RANGE = (1, 3) 
documents = {
    "source_1.txt": "Machine learning is a subset of artificial intelligence that focuses on data and algorithms.",
    "student_1.txt": "A subset of artificial intelligence, machine learning focuses on data and algorithms to imitate learning.",
    "source_2.txt": "Python is widely used in data science and web development for its simplicity.",
    "student_2.txt": "Data science and web development often use Python due to its simple syntax.",
    "blog_1.txt": "The stock market crashed yesterday due to inflation fears and rising rates."
}
def normalize_text(text):
    """Lowercase, remove punctuation, remove stopwords"""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    stop_words = set(stopwords.words('english'))
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)
def main():
    print("=== Task 3: Combat Online Plagiarism with AI ===\n")
    print("=== Normalized Texts ===")
    clean_docs = {}
    for name, text in documents.items():
        clean_docs[name] = normalize_text(text)
        print(f"{name}: {clean_docs[name]}")
    doc_names = list(clean_docs.keys())
    corpus = list(clean_docs.values())
    print("\n=== Generating N-gram TF-IDF Features ===")
    vectorizer = TfidfVectorizer(ngram_range=NGRAM_RANGE)
    tfidf_matrix = vectorizer.fit_transform(corpus)
    print(f"Total n-grams in vocabulary: {len(vectorizer.get_feature_names_out())}")
    cosine_sim = cosine_similarity(tfidf_matrix)
    results = []
    for i in range(len(doc_names)):
        for j in range(i + 1, len(doc_names)):
            cos_score = cosine_sim[i][j]
            fuzzy_score = fuzz.token_sort_ratio(corpus[i], corpus[j]) / 100
            
            if cos_score > THRESHOLD or fuzzy_score > THRESHOLD:
                results.append({
                    "Document 1": doc_names[i],
                    "Document 2": doc_names[j],
                    "Cosine Similarity": round(cos_score, 3),
                    "Fuzzy Score": round(fuzzy_score, 3),
                    "Verdict": "Potential Plagiarism"
                })
    print("\n=== Plagiarism Detection Report ===")
    if results:
        df_report = pd.DataFrame(results)
        print(df_report.to_string(index=False))
        df_report.to_csv("plagiarism_report.csv", index=False)
        print(f"\nReport saved to plagiarism_report.csv")
    else:
        print(f"No potential plagiarism found above {THRESHOLD} threshold.")
    print("\n=== Full Similarity Matrix ===")
    sim_df = pd.DataFrame(cosine_sim, index=doc_names, columns=doc_names)
    print(sim_df.round(3))
if __name__ == "__main__":
    main()

=== Task 3: Combat Online Plagiarism with AI ===

=== Normalized Texts ===
source_1.txt: machine learning subset artificial intelligence focuses data algorithms
student_1.txt: subset artificial intelligence machine learning focuses data algorithms imitate learning
source_2.txt: python widely used data science web development simplicity
student_2.txt: data science web development often use python due simple syntax
blog_1.txt: stock market crashed yesterday due inflation fears rising rates

=== Generating N-gram TF-IDF Features ===
Total n-grams in vocabulary: 92

=== Plagiarism Detection Report ===
  Document 1    Document 2  Cosine Similarity  Fuzzy Score              Verdict
source_1.txt student_1.txt              0.542        0.893 Potential Plagiarism
source_2.txt student_2.txt              0.312        0.760 Potential Plagiarism

Report saved to plagiarism_report.csv

=== Full Similarity Matrix ===
               source_1.txt  student_1.txt  source_2.txt  student_2.txt  \
source_1.